# Permuted neighboring accesses can still be coalesced

This notebook revisits `simpleExampleNonConsecutive.cu`. In that code, even lanes read the next element and odd lanes read the previous one. The access order is permuted in adjacent pairs, but each warp still touches a contiguous range. The two kernels below produce the same pairwise result: one permutes loads, the other permutes stores. Both are checked against a CPU reference and timed.

## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell shows the GPU and CUDA compiler available to the notebook.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov 14 15:06:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

`CUDA_CHECK` handles CUDA errors independently of `TimingGPU`. The timer uses CUDA events and reports elapsed milliseconds for GPU work.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}
TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}
void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}
float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## CUDA kernel and host program

For an even vector length, the original even/odd rule is equivalent to `i ^ 1`. For lanes 0–31, the source indices are 1, 0, 3, 2, and so on: they are not increasing, but they cover the same adjacent sectors. This is a lesson about address distribution, not an example of a coalescing penalty. The measured times also include index-generation and instruction costs.

In [5]:
%%writefile code/chapter02_permuted_coalesced.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cstdio>
#include <cstdlib>

const int N = 1 << 20;       // even, as required by the pair permutation
const int THREADS = 256;
const int REPEATS = 100;

// Each lane loads the neighboring element in its adjacent pair.
__global__ void pair_permuted_load(const int *a, const int *b, int *c, int n) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        const int source = i ^ 1;
        c[i] = a[source] + b[source];
    }
}

// Same result: lanes load consecutive elements and permute adjacent stores.
__global__ void pair_permuted_store(const int *a, const int *b, int *c, int n) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        const int destination = i ^ 1;
        c[destination] = a[i] + b[i];
    }
}

void launch_pair(bool permute_load, const int *a, const int *b, int *c, int n) {
    const int blocks = (n + THREADS - 1) / THREADS;
    if (permute_load)
        pair_permuted_load<<<blocks, THREADS>>>(a, b, c, n);
    else
        pair_permuted_store<<<blocks, THREADS>>>(a, b, c, n);
    CUDA_CHECK(cudaGetLastError());
}

int main() {
    const size_t bytes = size_t(N) * sizeof(int);
    int *h_a = (int*)std::malloc(bytes);
    int *h_b = (int*)std::malloc(bytes);
    int *h_cpu = (int*)std::malloc(bytes);
    int *h_load = (int*)std::malloc(bytes);
    int *h_store = (int*)std::malloc(bytes);
    int *d_a = nullptr, *d_b = nullptr, *d_load = nullptr, *d_store = nullptr;
    if (!h_a || !h_b || !h_cpu || !h_load || !h_store) {
        std::fprintf(stderr, "Host allocation failed.\n");
        return EXIT_FAILURE;
    }
    for (int i = 0; i < N; ++i) {
        h_a[i] = i % 127;
        h_b[i] = (i * 5) % 131;
    }
    for (int i = 0; i < N; ++i) h_cpu[i] = h_a[i ^ 1] + h_b[i ^ 1];

    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_load, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_store, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    launch_pair(true, d_a, d_b, d_load, N);
    launch_pair(false, d_a, d_b, d_store, N);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_load, d_load, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_store, d_store, bytes, cudaMemcpyDeviceToHost));
    for (int i = 0; i < N; ++i) {
        if (h_load[i] != h_cpu[i] || h_store[i] != h_cpu[i]) {
            std::fprintf(stderr, "CPU/GPU mismatch at i=%d.\n", i);
            return EXIT_FAILURE;
        }
    }
    std::printf("CPU vs permuted-load GPU: PASS (%d values)\n", N);
    std::printf("CPU vs permuted-store GPU: PASS (%d values)\n", N);

    TimingGPU timer;
    float elapsed[2];
    for (int method = 0; method < 2; ++method) {
        const bool permute_load = method == 0;
        int *d_out = permute_load ? d_load : d_store;
        for (int k = 0; k < 10; ++k) launch_pair(permute_load, d_a, d_b, d_out, N);
        CUDA_CHECK(cudaDeviceSynchronize());
        timer.StartCounter();
        for (int k = 0; k < REPEATS; ++k) launch_pair(permute_load, d_a, d_b, d_out, N);
        elapsed[method] = timer.GetCounter() / REPEATS;
    }
    std::printf("Permuted loads:  %.6f ms/launch (average of %d)\n", elapsed[0], REPEATS);
    std::printf("Permuted stores: %.6f ms/launch (average of %d)\n", elapsed[1], REPEATS);
    std::printf("Both lane mappings cover the same adjacent data sectors; timings also include index and instruction costs.\n");

    CUDA_CHECK(cudaFree(d_a)); CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_load)); CUDA_CHECK(cudaFree(d_store));
    std::free(h_a); std::free(h_b); std::free(h_cpu); std::free(h_load); std::free(h_store);
    return EXIT_SUCCESS;
}


Writing code/chapter02_permuted_coalesced.cu


## Compile and run

The program checks the GPU result against a CPU reference and prints timings averaged over repeated kernel launches. The timing excludes allocation, host-device copies, and validation. Measurements depend on the selected GPU.

In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_permuted_coalesced.cu code/TimingGPU.cu -o memory_example
./memory_example


CPU vs permuted-load GPU: PASS (1048576 values)
CPU vs permuted-store GPU: PASS (1048576 values)
Permuted loads:  0.051019 ms/launch (average of 100)
Permuted stores: 0.050892 ms/launch (average of 100)
Both lane mappings cover the same adjacent data sectors; timings also include index and instruction costs.
